# Fine-tuning XLS-R 300M для ASR


In [36]:
#!pip uninstall -y tqdm ipywidgets huggingface_hub

In [37]:
#!pip uninstall -y ipywidgets==8.1.2
#!pip install ipywidgets==7.7.5

In [38]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [39]:
!pip list | grep -E "transformers|huggingface|tqdm|ipywidgets|jupyter"

huggingface-hub           0.14.1
ipywidgets                7.7.5
jupyter_client            8.3.1
jupyter-console           6.4.0
jupyter_core              5.3.1
jupyter-events            0.7.0
jupyter-lsp               2.2.0
jupyter_server            2.7.3
jupyter_server_terminals  0.4.4
jupyterlab                4.2.5
jupyterlab-pygments       0.1.2
jupyterlab_server         2.27.3
jupyterlab_widgets        1.1.11
tqdm                      4.67.3
transformers              4.28.0


In [40]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

In [41]:
#!pip uninstall accelerate -y
#!pip install accelerate==0.19.0

In [42]:
import accelerate
print(accelerate.__version__)

0.19.0


In [43]:
#!pip install numpy==1.26.4
#!pip install scipy==1.11.4
#!pip install scikit-learn==1.3.2
#!pip uninstall transformers -y
#!pip install transformers==4.28.0
#!pip install huggingface_hub==0.14.1

In [44]:
#!pip uninstall torch -y
#!pip install torch==2.1.0+cu121 --index-url https://download.pytorch.org/whl/cu121

In [45]:
#!pip install jiwer

In [46]:
from pathlib import Path
import json
import random
import re
import wave
import unicodedata
from dataclasses import dataclass
from typing import Dict, List, Union

import IPython.display as ipd
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display, HTML

from jiwer import wer as jiwer_wer, cer as jiwer_cer

from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    Wav2Vec2CTCTokenizer,
    Trainer,
    TrainingArguments,
)

print("torch:", torch.__version__)
try:
    import transformers
    print("transformers:", transformers.__version__)
except Exception:
    pass

print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch: 2.1.0+cu121
transformers: 4.28.0
cuda available: True
gpu: Tesla V100-SXM2-32GB


In [47]:
#!pip install --upgrade numexpr==2.8.4 bottleneck==1.3.6

In [48]:
pip list | grep -E "numpy|scipy|scikit|transformers"

numpy                     1.26.4
scikit-learn              1.3.2
scipy                     1.11.4
transformers              4.28.0
Note: you may need to restart the kernel to use updated packages.


In [49]:
METADATA_CSV = pd.read_csv('metadata.csv')
METADATA_CSV['audio'] = METADATA_CSV['audio'].str.replace('/Users/air/Downloads/Ненецкие аудио/', '')
AUDIO_ROOT = Path("/data")

MODEL_NAME = "facebook/wav2vec2-xls-r-300m"
OUTPUT_DIR = Path("xls-r-300m-nenets-local")

AUDIO_COL = "audio"
TEXT_COL = "text"

TARGET_SAMPLING_RATE = 16000
TEST_SIZE = 0.2
SEED = 42

MAX_STEPS = 2000
NUM_TRAIN_EPOCHS = 5

In [50]:
rng = np.random.default_rng(SEED)
indices = np.arange(len(METADATA_CSV))
rng.shuffle(indices)

split_idx = int(len(indices) * (1 - TEST_SIZE))
train_idx = indices[:split_idx]
test_idx = indices[split_idx:]

nenets_train_df = METADATA_CSV.iloc[train_idx].reset_index(drop=True)
nenets_test_df = METADATA_CSV.iloc[test_idx].reset_index(drop=True)

print("train:", nenets_train_df.shape, "test:", nenets_test_df.shape)

train: (2669, 2) test: (668, 2)


In [51]:
def clean_text(text):
    text = re.sub(r"\s+", " ", text).strip()
    return text

nenets_train_df["transcription"] = nenets_train_df["transcription"].map(clean_text)
nenets_test_df["transcription"] = nenets_test_df["transcription"].map(clean_text)


def show_random_elements(df, num_examples=10):
    num_examples = min(num_examples, len(df))
    picks = random.sample(range(len(df)), num_examples)
    display(HTML(df.iloc[picks].to_html()))

show_random_elements(nenets_train_df[["audio", "transcription"]], num_examples=5)

,audio,transcription
2619,data/NENETS_13_322633.0_326667.0.wav,"ханедами ӈули"" яӈгу ӈули"" пэдэяв"""
456,data/Три ханты_new_879567.0859345581_883885.6036399907.wav,"ӈоб"" паӈгляр' хаби хонай' сидя ӈарка хаби маӈаха'"
102,data/NENETS_02_01_123041.9_130026.4.wav,"тайкэад нисяв мале еде""лыацьдакы в ом году ха яӈгумасьа"
1126,data/015_85401.59375_94581.59375.wav,"иудей"" параӈодаӈэ вадетамда"" ханзер"" мэӈгув пыдо' ӈани' тарем' тюры""ӈа"" лярць' пян' тебарт пилат ӈани' хонрэйда"
1003,data/Царь и Поп_new_669529.71875_674254.7187499999.wav,нюми яӈгорханё' тюку яля' варк' ха тюкона мэӈгусь юӈгу


In [52]:
all_text = " ".join(pd.concat([nenets_train_df["transcription"], nenets_test_df["transcription"]]).tolist())

vocab_list = sorted(set(all_text))
vocab_dict = {v: k for k, v in enumerate(vocab_list)}

if " " in vocab_dict:
    vocab_dict["|"] = vocab_dict[" "]
    del vocab_dict[" "]

vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

print("vocab size:", len(vocab_dict))
print(vocab_dict)

vocab size: 39
{'"': 1, "'": 2, 'а': 3, 'б': 4, 'в': 5, 'г': 6, 'д': 7, 'е': 8, 'ж': 9, 'з': 10, 'и': 11, 'й': 12, 'к': 13, 'л': 14, 'м': 15, 'н': 16, 'о': 17, 'п': 18, 'р': 19, 'с': 20, 'т': 21, 'у': 22, 'ф': 23, 'х': 24, 'ц': 25, 'ч': 26, 'ш': 27, 'щ': 28, 'ъ': 29, 'ы': 30, 'ь': 31, 'э': 32, 'ю': 33, 'я': 34, 'ё': 35, 'ӈ': 36, '|': 0, '[UNK]': 37, '[PAD]': 38}


In [53]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_DIR / "vocab.json", "w", encoding="utf-8") as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False, indent=2)

tokenizer = Wav2Vec2CTCTokenizer(
    str(OUTPUT_DIR / "vocab.json"),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=TARGET_SAMPLING_RATE,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

processor.save_pretrained(OUTPUT_DIR)
print(f"Processor saved to {OUTPUT_DIR}")

Processor saved to xls-r-300m-nenets-local


In [54]:
def load_wav_mono_16k(path: Path):
    path = Path(path)
    with wave.open(str(path), "rb") as wf:
        sample_rate = wf.getframerate()
        num_frames = wf.getnframes()
        audio_bytes = wf.readframes(num_frames)

    waveform = np.frombuffer(audio_bytes, dtype=np.int16).astype(np.float32)
    waveform = waveform / 32768.0
    waveform = torch.from_numpy(waveform)
    return waveform.numpy(), sample_rate

In [55]:
class LocalASRDataset(torch.utils.data.Dataset):
    def __init__(self, df: pd.DataFrame, processor: Wav2Vec2Processor):
        self.df = df.reset_index(drop=True)
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = row["audio"]
        speech_array, sampling_rate = load_wav_mono_16k(audio_path)

        processed = self.processor(
            speech_array,
            sampling_rate=sampling_rate,
        )

        labels = self.processor.tokenizer(row["transcription"]).input_ids

        return {
            "input_values": processed.input_values[0],
            "labels": labels,
        }


nenets_train = LocalASRDataset(nenets_train_df, processor)
nenets_test = LocalASRDataset(nenets_test_df, processor)

print("train samples:", len(nenets_train), "test samples:", len(nenets_test))

train samples: 2669 test samples: 668


In [56]:
rand_int = random.randint(0, len(nenets_train_df) - 1)
row = nenets_train_df.iloc[rand_int]
audio_path = row["audio"]
speech_array, sampling_rate = load_wav_mono_16k(audio_path)

print(row["transcription"])
print(audio_path, sampling_rate, speech_array.shape, speech_array.dtype)

ipd.Audio(data=speech_array, autoplay=False, rate=sampling_rate)

иисус нянда ма филиппэ нянанда" понхав' мэдамнё" пыдар тамна си"ми ехэран
data/014_66417.21875_72475.34375000001.wav 16000 (96930,) float32


In [57]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.pad(
            labels=label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch


data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

In [58]:
def compute_metrics(pred):
    print("compute_metrics started")

    pred_logits = pred.predictions
    print("logits shape:", pred_logits.shape)

    pred_ids = np.argmax(pred_logits, axis=-1)
    print("argmax done")

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    print("pred decode done")

    label_str = processor.batch_decode(
        label_ids,
        group_tokens=False
    )
    print("label decode done")

    wer = jiwer_wer(label_str, pred_str)
    print("wer done")

    cer = jiwer_cer(label_str, pred_str)
    print("cer done")

    return {"wer": wer, "cer": cer}

In [59]:
#!pip install ipywidgets
#%pip install ipywidgets
#%jupyter nbextension enable --py widgetsnbextension
#%jupyter nbextension install --py widgetsnbextension

In [60]:
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    attention_dropout=0.05,
    hidden_dropout=0.05,
    feat_proj_dropout=0.05,
    mask_time_prob=0.05,
    layerdrop=0.0,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True,
    force_download=False,
)

model.freeze_feature_encoder()

print("Loaded:", MODEL_NAME)

/home/user/.local/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")
Some weights of the model checkpoint at facebook/wav2vec2-xls-r-300m were not used when initializing Wav2Vec2ForCTC: ['quantizer.codevectors', 'project_hid.bias', 'project_q.weight', 'project_q.bias', 'quantizer.weight_proj.weight', 'quantizer.weight_proj.bias', 'project_hid.weight']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassific

Loaded: facebook/wav2vec2-xls-r-300m


In [61]:
print(torch.cuda.device_count())

4


In [62]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,

    evaluation_strategy="steps",

    max_steps=MAX_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,

    gradient_checkpointing=False,
    fp16=torch.cuda.is_available(),

    save_steps=200,
    eval_steps=200,
    logging_steps=1,

    learning_rate=3e-5,
    warmup_steps=100,

    save_total_limit=2,

    push_to_hub=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=nenets_train,
    eval_dataset=nenets_test,
)

In [63]:
trainer.train()

/home/user/.local/lib/python3.10/site-packages/transformers/optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
/home/user/.local/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Step,Training Loss,Validation Loss,Wer,Cer
200,3.384500,3.419684,1.000000,1.000000
400,3.165200,3.169595,1.000000,1.000000
600,2.636000,2.623410,1.000000,0.756981
800,1.428600,1.115183,0.789027,0.210749
1000,0.939900,0.838480,0.622730,0.156058
1200,0.737700,0.735022,0.545526,0.136379
1400,0.727700,0.704822,0.518746,0.128964
1600,0.758100,0.662282,0.489092,0.122166
1800,0.529900,0.643686,0.477988,0.119849
2000,0.604300,0.634474,0.477596,0.119501


compute_metrics started
logits shape: (668, 545, 39)
argmax done
pred decode done
label decode done
wer done
cer done


/home/user/.local/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


compute_metrics started
logits shape: (668, 545, 39)
argmax done
pred decode done
label decode done
wer done
cer done


...

/home/user/.local/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


compute_metrics started
logits shape: (668, 545, 39)
argmax done
pred decode done
label decode done
wer done
cer done


/home/user/.local/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


compute_metrics started
logits shape: (668, 545, 39)
argmax done
pred decode done
label decode done
wer done
cer done


TrainOutput(global_step=2000, training_loss=1.9116114318221809, metrics={'train_runtime': 1404.4437, 'train_samples_per_second': 45.57, 'train_steps_per_second': 1.424, 'total_flos': 1.9468349379498906e+19, 'train_loss': 1.9116114318221809, 'epoch': 23.81})

In [64]:
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print(f"Saved fine-tuned XLS-R model and processor to: {OUTPUT_DIR.resolve()}")

Saved fine-tuned XLS-R model and processor to: /home/user/xls-r-300m-nenets-local


In [65]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

input_dict = nenets_test[0]

with torch.no_grad():
    input_values = torch.tensor(input_dict["input_values"]).to(device).unsqueeze(0)
    logits = model(input_values).logits

pred_ids = torch.argmax(logits, dim=-1)[0]

print("prediction:", processor.decode(pred_ids))
labels = torch.tensor(input_dict["labels"])
print("reference :", processor.decode(labels, group_tokens=False).lower())

prediction: тикы вадида хэт" махаданда ӈарка" явна тёрей" лазарь тарпад"
reference : тикы вадида хэт" махаданда ӈарка" явана тёрей" лазарь тарпад"


In [66]:
predictions = []
references = []

for i in range(len(nenets_test)):
    input_dict = nenets_test[i]

    with torch.no_grad():
        input_values = torch.tensor(input_dict["input_values"]).to(device).unsqueeze(0)
        logits = model(input_values).logits

    pred_ids = torch.argmax(logits, dim=-1)[0]
    predictions.append(processor.decode(pred_ids))
    references.append(processor.decode(input_dict["labels"], group_tokens=False).lower())

wer_value = jiwer_wer(references, predictions)
cer_value = jiwer_cer(references, predictions)

print("WER:", wer_value)
print("CER:", cer_value)

pred_real = pd.DataFrame({"pred": predictions, "real": references})
display(pred_real.head(30))

WER: 0.4773350751143044
CER: 0.11944304970839288


,pred,real
0,"тикы вадида хэт"" махаданда ӈарка"" явна тёрей"" ...","тикы вадида хэт"" махаданда ӈарка"" явана тёрей""..."
1,"ненэць' еврей"" нылнава яля е""эмни ӈадиммы ни ӈ...","ненэць' еврей"" ныланава яля' е""эмня ӈадимы ни ..."
2,небяди' хар моӈаа' хар няби вэта еся хой нябив...,небянди' хар' моӈа' хар' няби вэта еся хой няб...
3,то' яду' сёларэй ӈэвы то'я' саляхна парэӈгода ...,то'яду' сё лыруй ӈэвы то'я саляна параӈгода' х...
4,маяанд' еремы ебрейм' самарий' тер нядада ӈабц...,маянд' еремы еврейм' самария' тер нядада ӈабца...
5,манзарана не таняна ӈани ситаманэць хэвханда м...,манзарана не таняна ӈани' сита манэць хэвханан...
6,"маня теневава"" нум' хэбяхасавэйха' ни инсели""","маня"" тенева нум' хэбяхасавэйха"" ни инзеле"""
7,"тедахав харто' сырмы ӈэбто' ӈод"" си""ми сэхэда""...","тедахав' харто' сырмы ӈэбто' ӈод"" си""ми сэхэда..."
8,мал' табсавэй варк' ха ибядулы',мал' табсавэй варк' хар ибедолы'
9,попанд' небяда го' ембэй' ся' мян харад' тер' ...,"попанд небядаӈо' емба'й' сямян харад' тер"" пар..."


In [67]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

WEIGHTS_PATH = OUTPUT_DIR / "model_state_dict.pt"
torch.save(model.state_dict(), WEIGHTS_PATH)

print("Local model directory:")
print(OUTPUT_DIR.resolve())
print("\nSaved weights file:")
print(WEIGHTS_PATH.resolve())

print("\nFiles in output directory:")
for file_path in sorted(OUTPUT_DIR.iterdir()):
    if file_path.is_file():
        size_mb = file_path.stat().st_size / (1024 ** 2)
        print(f"- {file_path.name} ({size_mb:.2f} MB)")

Local model directory:
/home/user/xls-r-300m-nenets-local

Saved weights file:
/home/user/xls-r-300m-nenets-local/model_state_dict.pt

Files in output directory:
- config.json (0.00 MB)
- model_state_dict.pt (1203.62 MB)
- preprocessor_config.json (0.00 MB)
- pytorch_model.bin (1203.60 MB)
- special_tokens_map.json (0.00 MB)
- tokenizer_config.json (0.00 MB)
- training_args.bin (0.00 MB)
- vocab.json (0.00 MB)


In [69]:
pred_real.to_csv('pred_real.csv')

In [71]:
!pip install Levenshtein

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.1/153.1 KB 1.5 MB/s eta 0:00:00a 0:00:01


In [72]:
from collections import Counter
import Levenshtein

confusions = Counter()

for ref, hyp in zip(references, predictions):
    ops = Levenshtein.opcodes(ref, hyp)

    for tag, i1, i2, j1, j2 in ops:
        if tag == "replace":
            for r, h in zip(ref[i1:i2], hyp[j1:j2]):
                confusions[(r, h)] += 1

In [73]:
for (ref_ch, pred_ch), cnt in confusions.most_common(50):
    print(f"{ref_ch} -> {pred_ch}: {cnt}")

" -> ': 153
' -> ": 69
о -> а: 61
и -> е: 61
д -> т: 57
а -> о: 50
е -> и: 47
а -> э: 44
у -> о: 41
е -> я: 40
о -> у: 35
э -> а: 32
я -> е: 26
и -> я: 25
ё -> ю: 24
а -> ': 22
я -> и: 21
я -> й: 19
а -> ы: 19
э -> ы: 19
ы -> э: 18
ы -> а: 17
з -> с: 17
ф -> п: 17
а ->  : 16
т -> д: 15
б -> п: 15
н -> ӈ: 14
я -> ь: 14
ь -> я: 13
я -> а: 13
  -> д: 12
ю -> ё: 11
' -> а: 11
и -> э: 10
' ->  : 10
ӈ -> м: 10
и -> ю: 10
н -> м: 10
в -> о: 10
м -> н: 10
у -> а: 9
м ->  : 9
а -> я: 9
а -> и: 9
в -> б: 8
о -> в: 8
  -> ӈ: 8
м -> ӈ: 8
о -> э: 8
